In [ ]:
import pandas as pd
import numpy as np

# ============================================================================
# 1. LOAD CLEANED DATA & PRE-PROCESS NEW TABLES
# ============================================================================
base_path = '..\\ACL_9th-F1\\F1_clean\\'

results = pd.read_csv('..\\ACL_9th-F1\\F1_clean\\results_clean_with_rank.csv')
races = pd.read_csv(base_path + 'races_clean.csv')
driver_standings = pd.read_csv(base_path + 'driver_standings_clean.csv')
qualifying = pd.read_csv('..\\ACL_9th-F1\\F1\\' + 'qualifying.csv')
drivers = pd.read_csv(base_path + 'drivers_clean.csv')
circuits = pd.read_csv(base_path + 'circuits_clean.csv')
constructors = pd.read_csv(base_path + 'constructors_clean.csv')
results['podium'] = (results['positionOrder'] <= 3).astype(int)
# Load the new tables
lap_times = pd.read_csv('..\\ACL_9th-F1\\F1\\lap_times.csv')
pit_stops = pd.read_csv('..\\ACL_9th-F1\\F1\\pit_stops.csv')
constructor_standings = pd.read_csv('..\\ACL_9th-F1\\F1\\constructor_standings.csv')

# Pre-process 'rank' (fastest lap rank) to numeric, converting '\N' to NaN
if 'rank' in results.columns:
    results['rank'] = pd.to_numeric(results['rank'].replace(r'\N', np.nan), errors='coerce')

# Aggregate average lap position per driver per race
race_lap_avg = lap_times.groupby(['raceId', 'driverId'])['position'].mean().reset_index(name='race_avg_lap_pos')

# Aggregate average pit stop duration per driver per race
pit_avg = pit_stops.groupby(['raceId', 'driverId'])['milliseconds'].mean().reset_index(name='race_avg_pit_ms')

# =============================================================================
# 2. DRIVER FORM FEATURES (Carries over between seasons)
# =============================================================================
# Merge results with races and new aggregates to sort chronologically
driver_form = results[['raceId', 'driverId', 'positionOrder', 'podium', 'rank']].copy()
driver_form = driver_form.merge(races[['raceId', 'race_year', 'race_round', 'circuitId']], on='raceId')
driver_form = driver_form.merge(race_lap_avg, on=['raceId', 'driverId'], how='left')
driver_form = driver_form.merge(pit_avg, on=['raceId', 'driverId'], how='left')

# Sort chronologically by driver
driver_form = driver_form.sort_values(by=['driverId', 'race_year', 'race_round'])

# --- RECENT FORM (Shifted to exclude current race) ---
driver_form['driver_prev_position'] = driver_form.groupby('driverId')['positionOrder'].shift(1)
driver_form['driver_prev_podium'] = driver_form.groupby('driverId')['podium'].shift(1).fillna(0)

g = driver_form.groupby('driverId')   # every calculation below stays inside one driver's own races

# Rolling 5-Race Averages (shift(1) = only earlier races)
driver_form['driver_avg_pos_last_5'] = g['positionOrder'].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
driver_form['driver_avg_lap_pos_last_5'] = g['race_avg_lap_pos'].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
driver_form['driver_avg_pit_ms_last_5'] = g['race_avg_pit_ms'].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
driver_form['driver_avg_speed_rank_last_5'] = g['rank'].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())

# --- CAREER TOTALS (all earlier races of this driver) ---
driver_form['driver_career_avg_pos'] = g['positionOrder'].transform(lambda s: s.shift(1).expanding().mean())
driver_form['driver_career_podiums'] = g['podium'].transform(lambda s: s.shift(1).cumsum()).fillna(0)
driver_form['driver_career_races'] = g.cumcount()        # number of EARLIER races (0 on debut)

# average finish at THIS circuit in earlier visits (shift(1) inside each driver-circuit pair)
gcirc = driver_form.groupby(['driverId', 'circuitId'])
driver_form['driver_avg_pos_at_circuit'] = gcirc['positionOrder'].transform(lambda s: s.shift(1).expanding().mean())

# Isolate columns for final merge
driver_features_to_keep = [
    'raceId', 'driverId', 'driver_prev_position', 'driver_prev_podium',
    'driver_avg_pos_last_5', 'driver_avg_lap_pos_last_5', 'driver_avg_pit_ms_last_5',
    'driver_avg_speed_rank_last_5', 'driver_career_avg_pos', 'driver_career_podiums',
    'driver_career_races', 'driver_avg_pos_at_circuit'
]
driver_form = driver_form[driver_features_to_keep]

# =============================================================================
# 3. DRIVER STANDINGS FEATURES (Resets to 0 every season)
# =============================================================================
ds = driver_standings.merge(races[['raceId', 'race_year', 'race_round']], on='raceId')
ds = ds.sort_values(by=['driverId', 'race_year', 'race_round'])

ds['pre_race_points'] = ds.groupby(['driverId', 'race_year'])['driver_points_cumm'].shift(1).fillna(0)
ds['pre_race_wins'] = ds.groupby(['driverId', 'race_year'])['driver_wins_cumm'].shift(1).fillna(0)
ds['won_previous_race'] = ds.groupby(['driverId', 'race_year'])['pre_race_wins'].diff().fillna(0)

ds = ds[['raceId', 'driverId', 'pre_race_points', 'pre_race_wins', 'won_previous_race']]

# =============================================================================
# 3.5 CONSTRUCTOR FORM & STANDINGS (NEW)
# =============================================================================
# --- CONSTRUCTOR FORM ---
constructor_form = results[['raceId', 'constructorId', 'positionOrder', 'podium']].copy()
constructor_form = constructor_form.merge(races[['raceId', 'race_year', 'race_round']], on='raceId')

# Best finish and total podiums for the team per race
constructor_form = constructor_form.groupby(['raceId', 'constructorId', 'race_year', 'race_round']).agg({
    'positionOrder': 'min',
    'podium': 'sum'
}).reset_index()

constructor_form = constructor_form.sort_values(by=['constructorId', 'race_year', 'race_round'])
gc = constructor_form.groupby('constructorId')   # every calculation stays inside one team's own races

constructor_form['constructor_prev_pos'] = gc['positionOrder'].shift(1)
constructor_form['constructor_avg_pos_last_5'] = gc['positionOrder'].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
constructor_form['constructor_career_podiums'] = gc['podium'].transform(lambda s: s.shift(1).cumsum()).fillna(0)

constructor_form = constructor_form[['raceId', 'constructorId', 'constructor_prev_pos', 'constructor_avg_pos_last_5', 'constructor_career_podiums']]

# --- CONSTRUCTOR STANDINGS ---
cs = constructor_standings.merge(races[['raceId', 'race_year', 'race_round']], on='raceId')
cs = cs.sort_values(by=['constructorId', 'race_year', 'race_round'])

# Adjust column names 'constructor_points_cumm'/'constructor_wins_cumm' if they differ in your CSV
cs['pre_race_constructor_points'] = cs.groupby(['constructorId', 'race_year'])['points'].shift(1).fillna(0)
cs['pre_race_constructor_wins'] = cs.groupby(['constructorId', 'race_year'])['wins'].shift(1).fillna(0)

cs = cs[['raceId', 'constructorId', 'pre_race_constructor_points', 'pre_race_constructor_wins']]

# =============================================================================
# 4. BUILD THE BASE TRAINING TABLE (Strict Leakage Control)
# =============================================================================
final_df = results[['raceId', 'driverId', 'constructorId', 'grid', 'podium']].copy()

final_df = final_df.merge(races[['raceId', 'race_year', 'race_round', 'circuitId']], on='raceId', how='left')

final_df = final_df.merge(qualifying[['raceId', 'driverId', 'position']], on=['raceId', 'driverId'], how='left')
final_df.rename(columns={'position': 'quali_position'}, inplace=True)

final_df = final_df.merge(drivers[['driverId', 'drivers_nationality']], on='driverId', how='left')
final_df.rename(columns={'drivers_nationality': 'driver_nationality'}, inplace=True)

final_df = final_df.merge(circuits[['circuitId', 'country']], on='circuitId', how='left')
final_df.rename(columns={'country': 'circuit_country'}, inplace=True)

final_df = final_df.merge(constructors[['constructorId', 'name']], on='constructorId', how='left')
final_df.rename(columns={'name': 'constructor_name'}, inplace=True)

final_df = final_df.merge(driver_form, on=['raceId', 'driverId'], how='left')
final_df = final_df.merge(ds, on=['raceId', 'driverId'], how='left')

# Add Constructor details
final_df = final_df.merge(constructor_form, on=['raceId', 'constructorId'], how='left')
final_df = final_df.merge(cs, on=['raceId', 'constructorId'], how='left')

# =============================================================================
# 5. HOME RACE FEATURE ENGINEERING
# =============================================================================
# Dictionary mapping F1 nationalities to circuit countries
nationality_to_country = {
    'British': 'UK', 'German': 'Germany', 'Spanish': 'Spain', 'Finnish': 'Finland',
    'Japanese': 'Japan', 'French': 'France', 'Polish': 'Poland', 'Brazilian': 'Brazil',
    'Italian': 'Italy', 'Australian': 'Australia', 'Austrian': 'Austria', 'American': 'USA',
    'Dutch': 'Netherlands', 'Colombian': 'Colombia', 'Portuguese': 'Portugal', 
    'Canadian': 'Canada', 'Indian': 'India', 'Hungarian': 'Hungary', 'Irish': 'Ireland',
    'Danish': 'Denmark', 'Argentine': 'Argentina', 'Belgian': 'Belgium', 
    'South African': 'South Africa', 'Swiss': 'Switzerland', 'Swedish': 'Sweden',
    'Mexican': 'Mexico', 'Russian': 'Russia', 'Chinese': 'China', 
    'Monegasque': 'Monaco', 'New Zealander': 'New Zealand'
}

final_df['driver_home_country'] = final_df['driver_nationality'].map(nationality_to_country).fillna(final_df['driver_nationality'])
# final_df['circuit_country'] = final_df['circuit_country'].replace({'United States': 'USA'})
final_df['is_home_race'] = (final_df['driver_home_country'] == final_df['circuit_country']).astype(int)
final_df = final_df.drop(columns=['driver_home_country'])

# =============================================================================
# 6. FINAL CLEANUP & EXPORT
# =============================================================================
# 6A. Grid position with grid 0 (pit-lane start / no grid slot) moved behind the last car of THAT race,
#     then use it to fill missing qualifying positions
last_slot = final_df.groupby('raceId')['grid'].transform('max')
final_df['grid_pos'] = final_df['grid'].where(final_df['grid'] != 0, last_slot + 1)
final_df['quali_position'] = final_df['quali_position'].fillna(final_df['grid_pos'])
# grid vs qualifying: positive = starts behind where he qualified (penalty)
final_df['grid_penalty_delta'] = final_df['grid_pos'] - final_df['quali_position']

# 1 = qualified ahead of his teammate(s), 0 = behind, 0.5 = no teammate in this race
team_rank = final_df.groupby(['raceId', 'constructorId'])['quali_position'].rank(method='first')
team_size = final_df.groupby(['raceId', 'constructorId'])['driverId'].transform('count')
final_df['outqualified_teammate'] = np.where(team_size > 1, (team_rank == 1).astype(float), 0.5)

# 6B. Fill zero-baselines for debuts
final_df['pre_race_points'] = final_df['pre_race_points'].fillna(0)
final_df['pre_race_wins'] = final_df['pre_race_wins'].fillna(0)
final_df['won_previous_race'] = final_df['won_previous_race'].fillna(0)
final_df['driver_prev_podium'] = final_df['driver_prev_podium'].fillna(0)
final_df['pre_race_constructor_points'] = final_df['pre_race_constructor_points'].fillna(0)
final_df['pre_race_constructor_wins'] = final_df['pre_race_constructor_wins'].fillna(0)

# 6C. Fill rookie/debut Form with "back of pack" baseline (20)
final_df['driver_prev_position'] = final_df['driver_prev_position'].fillna(20)
final_df['driver_avg_pos_last_5'] = final_df['driver_avg_pos_last_5'].fillna(20)
final_df['driver_career_avg_pos'] = final_df['driver_career_avg_pos'].fillna(20)
# first visit to a circuit: no history there yet -> use his overall career average instead
final_df['driver_avg_pos_at_circuit'] = final_df['driver_avg_pos_at_circuit'].fillna(final_df['driver_career_avg_pos'])
final_df['constructor_prev_pos'] = final_df['constructor_prev_pos'].fillna(20)
final_df['constructor_avg_pos_last_5'] = final_df['constructor_avg_pos_last_5'].fillna(20)

# 6D. Fill missing lap/pit stop data (due to decades of missing coverage)
final_df['driver_avg_speed_rank_last_5'] = final_df['driver_avg_speed_rank_last_5'].fillna(20)
final_df['driver_avg_lap_pos_last_5'] = final_df['driver_avg_lap_pos_last_5'].fillna(20)
# Fill missing pit stop milliseconds with a heavy penalty (e.g. 40000ms / 40s)
final_df['driver_avg_pit_ms_last_5'] = final_df['driver_avg_pit_ms_last_5'].fillna(40000)

# 6E. Championship points relative to the leader BEFORE this race (same 0-1 scale in every season)
leader = final_df.groupby('raceId')['pre_race_points'].transform('max')
final_df['points_share'] = np.where(leader > 0, final_df['pre_race_points'] / leader, 0)    # round 1: nobody has points -> 0

leader_c = final_df.groupby('raceId')['pre_race_constructor_points'].transform('max')
final_df['constructor_points_share'] = np.where(leader_c > 0, final_df['pre_race_constructor_points'] / leader_c, 0)

final_df.to_csv(base_path + 'FINAL_training_table.csv', index=False)
print(f"Final Training Table built successfully! Shape: {final_df.shape}")
print("Columns:", final_df.columns.tolist())


In [ ]:
full_df = final_df.copy()     # all seasons, kept for the coverage plot and the training-window experiment
print(full_df['race_year'].min(), full_df['race_year'].max(), full_df.shape)

### We are going to discard the years before 2004 due to the absence of qualifying systems in these years

In [ ]:
final_df = final_df[final_df['race_year'] > 2003]

In [ ]:
final_df.isnull().sum()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Features + target (grid_pos = grid with pit-lane starts moved behind the field)
selected_columns = ["podium",
                    # starting position
                    "quali_position", "grid_pos", "grid_penalty_delta", "outqualified_teammate",
                    # driver recent form
                    "driver_avg_pos_last_5", "driver_avg_lap_pos_last_5", "driver_avg_speed_rank_last_5",
                    "driver_avg_pit_ms_last_5", "driver_prev_position", "driver_prev_podium", "won_previous_race",
                    # driver career
                    "driver_career_avg_pos", "driver_career_podiums", "driver_career_races",
                    "driver_avg_pos_at_circuit", "is_home_race",
                    # driver standings
                    "pre_race_points", "pre_race_wins", "points_share",
                    # constructor form & standings
                    "constructor_prev_pos", "constructor_avg_pos_last_5", "constructor_career_podiums",
                    "pre_race_constructor_points", "pre_race_constructor_wins", "constructor_points_share"]

# 2. Calculate the correlation matrix
matrix = final_df[selected_columns].corr()

print("--- Correlation with podium (sorted) ---")
print(matrix['podium'].drop('podium').sort_values().round(3))

# 3. Heatmap
plt.figure(figsize=(22, 17))
sns.heatmap(matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f", annot_kws={'size': 7})
plt.title("Correlation Matrix: features vs podium (2004+)")
plt.tight_layout()
plt.show()

In [ ]:
final_df['race_year'].value_counts().sort_index().plot(kind='bar', figsize=(16, 4))
plt.title('Number of races per season')
plt.xlabel('season'); plt.ylabel('races')
plt.show()
print('first season:', final_df['race_year'].min(), '| last season:', final_df['race_year'].max(),
      '| seasons:', final_df['race_year'].nunique())

## Feature Selection

### 1. Reason for the chosen features (with evidence)
Every candidate must be **known before the race starts** (after qualifying). The evidence is the correlation with
`podium` on the modelling seasons (2004+, 8,650 rows, from the heatmap above) and the podium-rate plots below.

| Group | Feature | r with podium | Why it is relevant | Decision |
|---|---|---|---|---|
| Starting position | `quali_position` | −0.50 | Pace on the same weekend | Keep |
| | `grid` | −0.48 | Actual starting slot (after penalties) | Keep |
| Driver recent form | `driver_avg_lap_pos_last_5` | −0.48 | Average running position in the last 5 races | Keep |
| | `driver_avg_pos_last_5` | −0.47 | Average finish in the last 5 races | Keep |
| | `driver_avg_speed_rank_last_5` | −0.45 | Fastest-lap rank in the last 5 races (raw pace) | Keep |
| | `driver_prev_podium` | +0.41 | Podium in the previous race | Keep |
| | `driver_prev_position` | −0.35 | Finish in the previous race | Keep |
| | `won_previous_race` | +0.30 | Won the previous race | Keep |
| Driver career | `driver_career_avg_pos` | −0.38 | Long-term level of the driver | Keep |
| | `driver_career_podiums` | +0.29 | Proven podium history | Keep |
| Driver standings | `pre_race_points` | +0.43 | Season performance so far | Keep |
| | `pre_race_wins` | +0.43 | Season wins so far | Keep |
| Team form | `constructor_avg_pos_last_5` | −0.46 | Car strength over the last 5 races | Keep |
| | `constructor_prev_pos` | −0.40 | Team's best finish in the previous race | Keep |
| | `constructor_career_podiums` | +0.24 | Team's long-term history | Keep |
| Team standings | `pre_race_constructor_wins` | +0.45 | Team wins this season | Keep |
| | `pre_race_constructor_points` | +0.41 | Team points this season | Keep |
| Weak | `driver_avg_pit_ms_last_5` | +0.01 | Average pit-stop time in the last 5 races | **Drop** |
| | `is_home_race` | +0.00 | Racing in the driver's home country | **Drop** (kept for Q2 only) |

**Why the two features are dropped**
- `driver_avg_pit_ms_last_5`: pit-stop data only exists from 2011, so **all 2004–2010 rows (32% of the table)**
  hold the fill value 40,000 ms instead of a real time. On real values the relationship is weak and not one-directional
  (podium rate 21% → 10% → 16% from shortest to longest stops), and the linear correlation is +0.01.
- `is_home_race`: only 3.9% of rows are home races, and their podium rate is the same as everyone else's
  (14.7% vs 14.3%), which matches our answer to Q2.

**Redundancy (correlation between features ≥ 0.8)**: `pre_race_points` ~ `pre_race_constructor_points` (0.96),
`grid` ~ `quali_position` (0.93), `driver_avg_pos_last_5` ~ `driver_avg_lap_pos_last_5` (0.91). They are kept because
they still differ in meaningful cases (e.g. grid penalties: grid ≠ qualifying for a share of drivers; a driver can be
much better or worse than his team-mate). Tree models handle this overlap; we do not interpret logistic-regression
coefficients one by one.

### 2. Plotting the effect of the features
For each feature, the rows are split into 10 equal-size groups (deciles) from the lowest to the highest value, and
we plot the share of podiums in each group. A useful feature shows bars clearly above and below the red line
(overall podium rate, 14.3%).

In [ ]:
import matplotlib.pyplot as plt

FEATURES =[
    
    # --- Grid & Qualifying ---
    "quali_position", 
    "grid_pos", 
    "outqualified_teammate", 
    "grid_penalty_delta",
    
    # --- Driver Recent Form ---
    "driver_prev_position", 
    "driver_prev_podium",
    "driver_avg_pos_last_5", 
    "driver_avg_lap_pos_last_5", 
    "driver_avg_pit_ms_last_5", 
    "driver_avg_speed_rank_last_5",
    
    # --- Driver Career & Momentum ---
    "driver_career_avg_pos", 
    "driver_career_podiums", 
    "driver_career_races", 
    "driver_avg_pos_at_circuit",
    "is_home_race",
    
    # --- Driver Championship Standings ---
    "pre_race_points", 
    "pre_race_wins",
    "won_previous_race", 
    "points_share", 
    
    # --- Constructor (Car) Form & Standings ---
    "constructor_prev_pos",
    "constructor_avg_pos_last_5", 
    "constructor_career_podiums", 
    "pre_race_constructor_points",
    "pre_race_constructor_wins", 
    "constructor_points_share"
]

base_rate = final_df['podium'].mean()
fig, axes = plt.subplots(4, 5, figsize=(26, 17))
for ax, f in zip(axes.ravel(), FEATURES):
    d = final_df[[f, 'podium']]
    if d[f].nunique() <= 10:                       # few values: one bar per value
        rate = d.groupby(f)['podium'].mean()
        xlabel = 'value'
    else:                                          # many values: 10 equal-size groups (deciles), low -> high
        groups = pd.qcut(d[f], 10, duplicates='drop')
        rate = d.groupby(groups, observed=True)['podium'].mean()
        rate.index = range(1, len(rate) + 1)
        xlabel = 'decile (1 = lowest values)'
    rate.plot(kind='bar', ax=ax, color='steelblue')
    ax.axhline(base_rate, color='red', linestyle='--', linewidth=1)   # overall podium rate
    ax.set_title(f'{f}\nr = {final_df[f].corr(final_df["podium"]):+.2f}', fontsize=11)
    ax.set_xlabel(xlabel); ax.set_ylabel('podium rate'); ax.tick_params(axis='x', labelsize=7, rotation=0)
for ax in axes.ravel()[len(FEATURES):]:
    ax.axis('off')
plt.suptitle('Podium rate across the range of each feature (red line = overall podium rate)', fontsize=16)
plt.tight_layout(); plt.show()

**Observations**
- **Starting position is decisive:** the best decile of `grid` / `quali_position` reaches a ~65% podium rate, and from
  the 4th decile on it is almost 0.
- **Recent form behaves the same way:** the best decile of `driver_avg_pos_last_5`, `driver_avg_lap_pos_last_5`,
  `driver_avg_speed_rank_last_5` and `constructor_avg_pos_last_5` is at 55–60%, the worst near 0.
- **Yes/no features split clearly:** a podium in the previous race → ~50% podium rate (vs ~8%); winning the previous
  race → ~63% (vs ~12%).
- **Standings rise steadily:** the highest groups of `pre_race_points` / `pre_race_constructor_points` reach ~50%.
- **`driver_prev_position` rises slightly again at the worst values:** a retirement in the previous race gives a high
  position number even to fast drivers, so one bad race is a noisy signal — the 5-race average is cleaner.
- **`constructor_career_podiums` is not one-directional** (the 6th group is far above the 7th): a long history counts
  old successful teams that are no longer fast, so the recent team-form features carry the stronger signal.
- **`driver_avg_pit_ms_last_5` and `is_home_race` stay around the red line** → dropped (see the table above).

### 3. Leakage-safe form features (only past races)
The prediction is made after qualifying and before the race, so **no information from the race itself** may enter a
feature.

- **Excluded from features:** everything produced by the race being predicted (`positionOrder`, points, laps, status,
  that race's fastest-lap rank, lap times and pit stops).
- **Form and career features:** within each driver (or team), sorted by `race_year` and `race_round`, every value is
  computed from `shift(1)`, i.e. only from earlier races:
  `groupby('driverId')[col].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())`.
  The shift, the rolling window and the running totals all stay **inside the group**. An earlier version applied
  `.rolling()` / `.expanding()` after the grouping had ended, so the windows ran across drivers. This made
  `driver_career_podiums` correlate *negatively* with podium (−0.09); after the fix it is +0.29.
- **Standings features:** the cumulative points and wins are shifted by one race within the same driver/team and
  season, so they are the standings **before** the race (0 before the first race of a season).
- **Same-weekend information:** only `grid` and `quali_position`, both fixed before the start.
- **The season filter (2004+) is applied after the features are built**, so earlier seasons still count as history.

**Check:** for 200 random rows, the last-5 average finish and the career podium count are recomputed by hand from
strictly earlier races only, and compared with the table.

In [ ]:
# Leakage check: recompute two form features by hand for random rows, using ONLY races that happened earlier
hist = results[['raceId', 'driverId', 'positionOrder', 'podium']].merge(races[['raceId', 'race_year', 'race_round']], on='raceId')
hist['order'] = hist['race_year'] * 100 + hist['race_round']                  # time order: season, then round

sample = final_df.sample(200, random_state=42)
checks = []
for _, row in sample.iterrows():
    now = row['race_year'] * 100 + row['race_round']
    past = hist[(hist['driverId'] == row['driverId']) & (hist['order'] < now)].sort_values('order')   # strictly earlier races
    exp_avg5 = past['positionOrder'].tail(5).mean() if len(past) else 20       # 20 = notebook's fill for debuts
    exp_career = past['podium'].sum()
    checks.append({'avg_pos_last_5 matches': np.isclose(row['driver_avg_pos_last_5'], exp_avg5),
                   'career_podiums matches': row['driver_career_podiums'] == exp_career})
checks = pd.DataFrame(checks)
print('rows checked:', len(checks))
print(checks.mean().rename('share matching'))

**Result:** 200 / 200 rows match for both features (share = 1.0). Every value is computed only from races before the
one being predicted, so the form features are leakage-safe.

## Model training
### Train / validation / test split
The data is split **by season**, in time order: the models learn from past seasons and are judged on later ones,
exactly how they would be used for real (predicting the next races). A random split would put races of the same
season in train and test, letting the model "see the future".
- **Train 2004–2019:** all seasons with complete data up to the validation period.
- **Validation 2020–2021:** used to choose hyperparameters and the probability threshold.
- **Test 2022–2024:** touched only once, at the end, for the final scores.
Whole seasons stay together, so no race is split across sets.

In [ ]:
MODEL_FEATURES =[
    
    # --- Grid & Qualifying ---
    "quali_position", 
    "grid_pos", 
    "outqualified_teammate", 
    "grid_penalty_delta",
    
    # --- Driver Recent Form ---
    "driver_prev_position", 
    "driver_prev_podium",
    "driver_avg_pos_last_5", 
    "driver_avg_lap_pos_last_5", 
    "driver_avg_pit_ms_last_5", 
    "driver_avg_speed_rank_last_5",
    
    # --- Driver Career & Momentum ---
    "driver_career_avg_pos", 
    "driver_career_podiums", 
    "driver_career_races", 
    "driver_avg_pos_at_circuit",
    "is_home_race",
    
    # --- Driver Championship Standings ---
    "pre_race_points", 
    "pre_race_wins",
    "won_previous_race", 
    "points_share", 
    
    # --- Constructor (Car) Form & Standings ---
    "constructor_prev_pos",
    "constructor_avg_pos_last_5", 
    "constructor_career_podiums", 
    "pre_race_constructor_points",
    "pre_race_constructor_wins", 
    "constructor_points_share"
]
final_df=pd.read_csv('..\\ACL_9th-F1\\F1_clean\\all_features.csv')

# Split by SEASON (time order): learn from the past, predict the future
train_df = final_df[final_df['race_year'] <= 2019]
val_df   = final_df[final_df['race_year'].between(2020, 2021)]
test_df  = final_df[final_df['race_year'] >= 2022]

X_train, y_train = train_df[MODEL_FEATURES], train_df['podium']
X_val,   y_val   = val_df[MODEL_FEATURES],   val_df['podium']
X_test,  y_test  = test_df[MODEL_FEATURES],  test_df['podium']

split_summary = pd.DataFrame({
    'seasons': [f"{d['race_year'].min()}-{d['race_year'].max()}" for d in (train_df, val_df, test_df)],
    'races':   [d['raceId'].nunique() for d in (train_df, val_df, test_df)],
    'rows':    [len(d) for d in (train_df, val_df, test_df)],
    'podium rate': [round(d['podium'].mean(), 3) for d in (train_df, val_df, test_df)]},
    index=['train', 'validation', 'test'])
split_summary

### Shared evaluation tools
- **Metrics:** ROC-AUC and PR-AUC judge how well the model *ranks* podium finishers above the rest (PR-AUC is the main
  one because podiums are rare, ~15%). F1, precision and recall need a yes/no decision, so they depend on a threshold.
- **Threshold:** the probability cut-off that gives the best F1 **on validation**; never chosen on test.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, precision_recall_curve

def best_threshold(y, prob):
    # the probability cut-off that gives the highest F1 -- chosen on VALIDATION only
    precision, recall, thresholds = precision_recall_curve(y, prob)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    return thresholds[np.argmax(f1[:-1])]

def scores(y, prob, threshold):
    pred = (prob >= threshold).astype(int)
    return {'ROC-AUC': roc_auc_score(y, prob), 'PR-AUC': average_precision_score(y, prob),
            'F1': f1_score(y, pred), 'precision': precision_score(y, pred, zero_division=0),
            'recall': recall_score(y, pred)}

results = []          # one row per model, filled as we go
def record(name, prob_train, prob_val, threshold):
    row = {'model': name, 'threshold': round(float(threshold), 3)}
    row.update({f'train {k}': v for k, v in scores(y_train, prob_train, threshold).items()})
    row.update({f'val {k}': v for k, v in scores(y_val, prob_val, threshold).items()})
    results.append(row)
    return pd.DataFrame(results).set_index('model')[['threshold', 'train ROC-AUC', 'train PR-AUC', 'train F1',
                                                     'val ROC-AUC', 'val PR-AUC', 'val F1']].round(3)

### Baseline: "starting in the top 3 means a podium"
Every model must beat this simple rule; otherwise the extra features and the model add nothing.

In [ ]:
# Baseline rule: "a driver starting in the top 3 finishes on the podium"
# score = 1 / grid_pos (higher = better start); cut-off 1/3 means grid_pos <= 3
rule_train = 1 / train_df['grid_pos']
rule_val   = 1 / val_df['grid_pos']
record('Baseline: grid <= 3', rule_train, rule_val, threshold=1/3)

### Model 1: Logistic regression
A linear model: it adds up the features with weights and turns the sum into a probability. Simple and hard to
overfit, so it is a strong first model. Features are standardised (scaler fit on train only). The regularisation
strength `C` is chosen on validation PR-AUC.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Try a few regularisation strengths (C); choose the one with the best VALIDATION PR-AUC
lr_search = []
for C in [0.01, 0.1, 1, 10]:
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000))   # scaler is fit on TRAIN only
    m.fit(X_train, y_train)
    lr_search.append({'C': C, 'val PR-AUC': average_precision_score(y_val, m.predict_proba(X_val)[:, 1])})
lr_search = pd.DataFrame(lr_search)
print(lr_search.round(4).to_string(index=False))
best_C = lr_search.loc[lr_search['val PR-AUC'].idxmax(), 'C']
print('chosen C =', best_C)

lr_model = make_pipeline(StandardScaler(), LogisticRegression(C=best_C, max_iter=3000)).fit(X_train, y_train)
p_train_lr, p_val_lr = lr_model.predict_proba(X_train)[:, 1], lr_model.predict_proba(X_val)[:, 1]
record('Logistic regression', p_train_lr, p_val_lr, best_threshold(y_val, p_val_lr))

### Model 2: Random forest
Many decision trees, each trained on a random sample of the rows and features; the forest averages their votes.
Trees do not need scaled features. `min_samples_leaf` (the smallest group of rows a leaf may hold) controls how
detailed the trees get: too small and they memorise the training data. It is chosen on validation PR-AUC.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Random forest (Lab 4): many decision trees, each trained on a random sample of rows/features; their votes are averaged.
# Trees do not need scaled features, so the unscaled X is used.
# min_samples_leaf controls how detailed each tree may get (small = can memorise training rows = overfitting)
rf_search = []
for leaf in [1, 5, 20, 50]:
    m = RandomForestClassifier(n_estimators=300, min_samples_leaf=leaf, random_state=42, n_jobs=-1).fit(X_train, y_train)
    rf_search.append({'min_samples_leaf': leaf,
                      'train PR-AUC': average_precision_score(y_train, m.predict_proba(X_train)[:, 1]),
                      'val PR-AUC': average_precision_score(y_val, m.predict_proba(X_val)[:, 1])})
rf_search = pd.DataFrame(rf_search)
print(rf_search.round(4).to_string(index=False))
best_leaf = int(rf_search.loc[rf_search['val PR-AUC'].idxmax(), 'min_samples_leaf'])
print('chosen min_samples_leaf =', best_leaf)

rf_model = RandomForestClassifier(n_estimators=300, min_samples_leaf=best_leaf, random_state=42, n_jobs=-1).fit(X_train, y_train)
p_train_rf = rf_model.predict_proba(X_train)[:, 1]
p_val_rf   = rf_model.predict_proba(X_val)[:, 1]
thr_rf = best_threshold(y_val, p_val_rf)
record('Random forest', p_train_rf, p_val_rf, thr_rf)

### Model 3: Gradient boosting
Trees are built one after another, each correcting the mistakes of the previous ones. It overfits quickly on this
data, so it uses a small learning rate, a minimum of 100 rows per leaf and L2 regularisation; tree size and number of
trees are chosen on validation PR-AUC.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

# Gradient boosting: trees are built one after another, each one correcting the mistakes of the previous ones.
# max_leaf_nodes = how complex each tree is; max_iter = how many trees; min_samples_leaf = smallest group a leaf may hold.
# Small learning rate + regularisation, because boosting overfits this data quickly (train score rises, val score falls).
gb_search = []
for leaves in [3, 7, 15]:
    for n_trees in [100, 200, 400]:
        m = HistGradientBoostingClassifier(learning_rate=0.03, max_leaf_nodes=leaves, max_iter=n_trees,
                                           min_samples_leaf=100, l2_regularization=1.0, random_state=42).fit(X_train, y_train)
        gb_search.append({'max_leaf_nodes': leaves, 'trees': n_trees,
                          'train PR-AUC': average_precision_score(y_train, m.predict_proba(X_train)[:, 1]),
                          'val PR-AUC': average_precision_score(y_val, m.predict_proba(X_val)[:, 1])})
gb_search = pd.DataFrame(gb_search)
print(gb_search.round(4).to_string(index=False))
best_gb = gb_search.loc[gb_search['val PR-AUC'].idxmax()]
gb_params = dict(learning_rate=0.03, max_leaf_nodes=int(best_gb['max_leaf_nodes']), max_iter=int(best_gb['trees']),
                 min_samples_leaf=100, l2_regularization=1.0, random_state=42)
print('chosen:', gb_params)

gb_model = HistGradientBoostingClassifier(**gb_params).fit(X_train, y_train)
p_train_gb = gb_model.predict_proba(X_train)[:, 1]
p_val_gb   = gb_model.predict_proba(X_val)[:, 1]
thr_gb = best_threshold(y_val, p_val_gb)
record('Gradient boosting', p_train_gb, p_val_gb, thr_gb)

## Accuracy

In [ ]:
from sklearn.metrics import accuracy_score

# Accuracy = share of ALL rows (podium and no podium) predicted correctly, using each model's validation threshold.
# Test probabilities are only scored here -- nothing is chosen from them.
thr_lr = best_threshold(y_val, p_val_lr)          # the old LR cell did not save its threshold
acc_models = {
    'Always "no podium" (reference)': (np.zeros(len(y_train)), np.zeros(len(y_val)), np.zeros(len(y_test)), 0.5),
    'Baseline: grid <= 3':            (1 / train_df['grid_pos'], 1 / val_df['grid_pos'], 1 / test_df['grid_pos'], 1/3),
    'Logistic regression':            (p_train_lr, p_val_lr, lr_model.predict_proba(X_test)[:, 1], thr_lr),
    'Random forest':                  (p_train_rf, p_val_rf, rf_model.predict_proba(X_test)[:, 1], thr_rf),
    'Gradient boosting':              (p_train_gb, p_val_gb, gb_model.predict_proba(X_test)[:, 1], thr_gb),
}
acc_rows = []
for name, (p_tr, p_va, p_te, thr) in acc_models.items():
    acc_rows.append({'model': name,
                     'train accuracy': accuracy_score(y_train, (np.asarray(p_tr) >= thr).astype(int)),
                     'val accuracy':   accuracy_score(y_val,   (np.asarray(p_va) >= thr).astype(int)),
                     'test accuracy':  accuracy_score(y_test,  (np.asarray(p_te) >= thr).astype(int))})
pd.DataFrame(acc_rows).set_index('model').round(3)

**Accuracy is not used to choose models.** 85% of rows are non-podiums, so predicting "no podium" for everyone is
already 85% accurate. All models reach ~91%, barely above the grid rule, while PR-AUC (which focuses on the podium
class) separates them clearly: 0.64 for the rule vs 0.73–0.75 for the models.

## Final evaluation on the test set (2022–2024)
Every model is scored once on the unseen test seasons, using the hyperparameters and threshold it received from
validation. Nothing is chosen or changed based on these numbers.

In [ ]:
# FINAL TEST: every model, scored once on the unseen 2022-2024 seasons.
# Nothing is chosen here: each model keeps the settings and threshold it got from validation.
thr_lr = best_threshold(y_val, p_val_lr)          # same threshold the old LR cell used (it didn't save it)

test_probs = {
    'Baseline: grid <= 3': (1 / test_df['grid_pos'],                  1/3),
    'Logistic regression': (lr_model.predict_proba(X_test)[:, 1],      thr_lr),   # pipeline scales X_test itself
    'Random forest':       (rf_model.predict_proba(X_test)[:, 1],      thr_rf),
    'Gradient boosting':   (gb_model.predict_proba(X_test)[:, 1],      thr_gb),
}
# when the neural network exists, add:  'Neural network': (nn_model.predict(X_test_nn, verbose=0).ravel(), thr_nn)

final_table = []
for name, (prob, thr) in test_probs.items():
    row = next(r for r in results if r['model'] == name).copy()          # train + validation scores from before
    row.update({f'test {k}': v for k, v in scores(y_test, prob, thr).items()})
    final_table.append(row)
final_table = pd.DataFrame(final_table).set_index('model')
final_table[['threshold', 'train PR-AUC', 'val PR-AUC', 'test PR-AUC', 'train ROC-AUC', 'val ROC-AUC', 'test ROC-AUC',
             'train F1', 'val F1', 'test F1']].round(3)

### Model 4: Neural network (Keras)
A shallow fully connected network (Lab 3): inputs → 1–2 hidden layers with ReLU → one sigmoid output giving the
podium probability. Loss: binary cross-entropy (binary task); optimiser: Adam. Inputs are standardised (scaler fit on
train only). Early stopping watches the validation loss and keeps the best weights, so the network stops before it
starts memorising the training seasons. Two architectures are compared on validation PR-AUC.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'             # run Keras on PyTorch (TensorFlow cannot import on Python 3.9.0)
import keras
from keras import layers
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler

# Neural networks need scaled inputs (Lab 3): learn mean/std on TRAIN only
nn_scaler = StandardScaler().fit(X_train)
X_train_nn, X_val_nn, X_test_nn = nn_scaler.transform(X_train), nn_scaler.transform(X_val), nn_scaler.transform(X_test)

def build_model(hidden_units, dropout):
    keras.utils.set_random_seed(42)                       # same starting weights every run -> reproducible
    model = keras.Sequential([layers.Input(shape=(X_train_nn.shape[1],))])
    for units in hidden_units:                            # 1-2 hidden layers = shallow, as the brief asks
        model.add(layers.Dense(units, activation='relu'))
        if dropout > 0:
            model.add(layers.Dropout(dropout))            # randomly switches neurons off while training -> less overfitting
    model.add(layers.Dense(1, activation='sigmoid'))      # one output: probability of a podium (binary task)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='binary_crossentropy',
                  metrics=[keras.metrics.AUC(curve='PR', name='pr_auc')])
    return model

# Early stopping (Lab 3): stop when the validation loss has not improved for 15 epochs, keep the best weights
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

# Two architectures, chosen on validation PR-AUC
nn_search, histories, nn_models = [], {}, {}
for name, units, dropout in [('NN-1: 16 units', [16], 0.0), ('NN-2: 32-16 units + dropout', [32, 16], 0.2)]:
    model = build_model(units, dropout)
    hist = model.fit(X_train_nn, y_train, validation_data=(X_val_nn, y_val),
                     epochs=200, batch_size=64, callbacks=[early_stop], verbose=0)
    histories[name], nn_models[name] = hist, model
    nn_search.append({'network': name, 'epochs run': len(hist.history['loss']),
                      'train PR-AUC': average_precision_score(y_train, model.predict(X_train_nn, verbose=0).ravel()),
                      'val PR-AUC': average_precision_score(y_val, model.predict(X_val_nn, verbose=0).ravel())})
nn_search = pd.DataFrame(nn_search)
print(nn_search.round(4).to_string(index=False))
best_nn = nn_search.loc[nn_search['val PR-AUC'].idxmax(), 'network']
print('chosen:', best_nn)

nn_model = nn_models[best_nn]
p_train_nn = nn_model.predict(X_train_nn, verbose=0).ravel()
p_val_nn   = nn_model.predict(X_val_nn, verbose=0).ravel()
thr_nn = best_threshold(y_val, p_val_nn)
record('Neural network', p_train_nn, p_val_nn, thr_nn)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Neural network: yes/no decisions with the threshold chosen on validation
p_test_nn = nn_model.predict(X_test_nn, verbose=0).ravel()
pred_val_nn  = (p_val_nn  >= thr_nn).astype(int)
pred_test_nn = (p_test_nn >= thr_nn).astype(int)

labels = ['no podium', 'podium']
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (title, y_true, y_pred) in zip(axes, [('Validation 2020-2021', y_val, pred_val_nn),
                                              ('Test 2022-2024', y_test, pred_test_nn)]):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax, cbar=False)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(f'Neural network - {title}')
plt.tight_layout(); plt.show()

print(f'threshold = {thr_nn:.3f} (chosen on validation)\n')
print('TEST classification report:')
print(classification_report(y_test, pred_test_nn, target_names=labels))

In [ ]:
# Training curves (Lab 3): loss on train vs validation for each network
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (name, hist) in zip(axes, histories.items()):
    ax.plot(hist.history['loss'], label='train loss')
    ax.plot(hist.history['val_loss'], label='validation loss')
    ax.axvline(np.argmin(hist.history['val_loss']), color='grey', linestyle='--', label='best epoch (weights kept)')
    ax.set_title(name); ax.set_xlabel('epoch'); ax.set_ylabel('binary cross-entropy'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
    'Neural network':      (nn_model.predict(X_test_nn, verbose=0).ravel(), thr_nn),

#### Experiment: class weights for the imbalance
Only ~15% of rows are podiums. Stratification (as in Lab 3's `train_test_split`) is not needed, because the split is
by season and the podium rate is already equal across train/validation/test (14–15%, see the split table). Instead we
test **balanced class weights**, which make every podium row count ~3.6× and every non-podium row ~0.6× during
training. Each setting is trained with 3 random starts, because a single run can be lucky.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# 'balanced' weights: each class gets total weight as if both classes were equally common
weights = compute_class_weight(class_weight='balanced', classes=np.array([0, 1]), y=y_train)
class_weights = {0: weights[0], 1: weights[1]}
print('class weights:', {k: round(v, 2) for k, v in class_weights.items()})

cw_rows = []
for use_weights in [False, True]:
    for name, units, dropout in [('NN-1: 16 units', [16], 0.0), ('NN-2: 32-16 units + dropout', [32, 16], 0.2)]:
        for seed in [42, 7, 123]:                      # 3 random starts, because one run can be lucky
            keras.utils.set_random_seed(seed)
            m = build_model(units, dropout)
            keras.utils.set_random_seed(seed)
            h = m.fit(X_train_nn, y_train, validation_data=(X_val_nn, y_val), epochs=200, batch_size=64,
                      class_weight=class_weights if use_weights else None,
                      callbacks=[EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)], verbose=0)
            p = m.predict(X_val_nn, verbose=0).ravel()
            thr = best_threshold(y_val, p)
            cw_rows.append({'class weights': 'balanced' if use_weights else 'none', 'network': name, 'seed': seed,
                            'val PR-AUC': average_precision_score(y_val, p),
                            'val F1': scores(y_val, p, thr)['F1'], 'threshold': thr, 'mean predicted prob': p.mean()})
cw = pd.DataFrame(cw_rows)
cw.groupby(['network', 'class weights'])[['val PR-AUC', 'val F1', 'threshold', 'mean predicted prob']].mean().round(3)

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# The experiment loop does not keep its models, so train the class-weighted version once more:
# same architecture as the chosen network (NN-2), same seed, only class_weight added.
weights = compute_class_weight(class_weight='balanced', classes=np.array([0, 1]), y=y_train)
class_weights = {0: weights[0], 1: weights[1]}

nn_weighted = build_model([32, 16], 0.2)
nn_weighted.fit(X_train_nn, y_train, validation_data=(X_val_nn, y_val), epochs=200, batch_size=64,
                class_weight=class_weights,
                callbacks=[EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)], verbose=0)
p_val_w  = nn_weighted.predict(X_val_nn, verbose=0).ravel()
p_test_w = nn_weighted.predict(X_test_nn, verbose=0).ravel()
thr_w = best_threshold(y_val, p_val_w)                         # threshold chosen on validation, as for every model

# Side by side on the TEST seasons: network without vs with class weights
p_test_nn = nn_model.predict(X_test_nn, verbose=0).ravel()
compare = {'Neural network (no class weights)': (p_test_nn, thr_nn),
           'Neural network (balanced class weights)': (p_test_w, thr_w)}
labels = ['no podium', 'podium']
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (name, (prob, thr)) in zip(axes, compare.items()):
    cm = confusion_matrix(y_test, (prob >= thr).astype(int))
    tn, fp, fn, tp = cm.ravel()
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax, cbar=False,
                annot_kws={'size': 14})
    ax.set_title(f'{name}\nthreshold {thr:.2f} | precision {tp/(tp+fp):.2f} | recall {tp/(tp+fn):.2f}', fontsize=11)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('TEST seasons 2022-2024', fontsize=13)
plt.tight_layout(); plt.show()

for name, (prob, thr) in compare.items():
    s = scores(y_test, prob, thr)
    print(f"{name:42s} test PR-AUC {s['PR-AUC']:.3f} | ROC-AUC {s['ROC-AUC']:.3f} | F1 {s['F1']:.3f}")

**Observations:** qualifying is complete from 2003 (patchy 1994–2002), lap times from 1996, fastest-lap rank only
from 2004, pit stops only from 2011. 2004 is the first season in which every main feature has real data; earlier rows
hold fill values (e.g. 20 for the speed-rank feature).

### Training-window experiment
Each start year marks a change in the data or the rules. The models use fixed standard settings (tuning comes later,
on the chosen window), so the comparison is not circular.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

# The model's feature list (21) -- defined ONCE here and reused by the window experiment, the models and the ablation
MODEL_FEATURES = ["quali_position", "grid_pos", "driver_avg_pos_last_5", "driver_avg_lap_pos_last_5",
                  "driver_avg_speed_rank_last_5", "driver_prev_position", "driver_prev_podium",
                  "driver_career_avg_pos", "driver_career_podiums", "points_share", "pre_race_wins",
                  "won_previous_race", "constructor_prev_pos", "constructor_avg_pos_last_5",
                  "constructor_career_podiums", "constructor_points_share", "pre_race_constructor_wins",
                  "outqualified_teammate", "grid_penalty_delta", "driver_career_races", "driver_avg_pos_at_circuit"]
WINDOW_FEATURES = MODEL_FEATURES

# Candidate training windows: each start year marks a change in the data or the rules. Training always ends in 2019.
WINDOWS = {'1950+ (all seasons)':               1950,
           '1994+ (qualifying data starts)':     1994,
           '1996+ (lap times start)':            1996,
           '2003+ (qualifying complete)':        2003,
           '2004+ (fastest-lap rank starts)':    2004,
           '2011+ (pit stops start)':            2011,
           '2012+ (last 8 seasons)':             2012,
           '2014+ (hybrid era rules)':           2014}

# Fixed, standard settings (not tuned): tuning happens later, on the chosen window
MODELS = {'Logistic regression': lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
          'Gradient boosting':   lambda: HistGradientBoostingClassifier(learning_rate=0.05, max_leaf_nodes=15,
                                                                        l2_regularization=1.0, random_state=42)}

val_w = full_df[full_df['race_year'].between(2020, 2021)]          # the same validation seasons as the models later
window_rows, window_preds = [], {}
for wname, start in WINDOWS.items():
    tr = full_df[full_df['race_year'].between(start, 2019)]
    for mname, make in MODELS.items():
        prob = make().fit(tr[WINDOW_FEATURES], tr['podium']).predict_proba(val_w[WINDOW_FEATURES])[:, 1]
        window_preds[(wname, mname)] = prob
        window_rows.append({'training window': wname, 'model': mname, 'training rows': len(tr),
                            'val PR-AUC': average_precision_score(val_w['podium'], prob),
                            'val ROC-AUC': roc_auc_score(val_w['podium'], prob)})
window_results = pd.DataFrame(window_rows)
window_results.pivot(index='training window', columns='model', values='val PR-AUC').loc[list(WINDOWS)].round(3)

### Training-window comparison: all models, same validation and test seasons
Each model is trained on a different range of seasons (always ending in 2019) and evaluated on the **same**
validation (2020–2021) and test (2022–2024) seasons, so only the training data changes.
- **Models:** logistic regression, random forest, gradient boosting, and a shallow neural network
  (2 hidden layers: 32 and 16 units, ReLU, dropout 0.2, early stopping on validation loss).
- **Metrics:** PR-AUC (primary, podiums are rare), ROC-AUC and F1, on train, validation and test.
- **F1 threshold:** chosen on validation for each model, then applied unchanged to train and test.
- **Choice rule:** the window is chosen on **validation** scores; test scores are reported for reference only.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'            # Keras on PyTorch (works locally and on Kaggle)
import keras
from keras import layers
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_recall_curve

# ---- Fixed parts: the SAME validation and test seasons for every training window ----
val_set  = full_df[full_df['race_year'].between(2020, 2021)]
test_set = full_df[full_df['race_year'].between(2022, 2024)]

WINDOWS = {'1950+': 1950, '1994+': 1994, '1996+': 1996, '2003+': 2003,
           '2004+': 2004, '2011+': 2011, '2012+': 2012, '2014+': 2014}


def best_threshold(y, prob):
    """Probability cut-off with the highest F1 -- chosen on VALIDATION only."""
    precision, recall, thresholds = precision_recall_curve(y, prob)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    return thresholds[np.argmax(f1[:-1])]


def neural_network(n_features):
    """Shallow network: 2 hidden layers (32, 16) with ReLU, dropout, sigmoid output."""
    keras.utils.set_random_seed(42)
    model = keras.Sequential([layers.Input(shape=(n_features,)),
                              layers.Dense(32, activation='relu'), layers.Dropout(0.2),
                              layers.Dense(16, activation='relu'), layers.Dropout(0.2),
                              layers.Dense(1, activation='sigmoid')])
    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model


def train_and_predict(model_name, X_tr, y_tr, X_va, y_va, X_te):
    """Train one model; return its podium probabilities on train, validation and test."""
    if model_name in ('Logistic regression', 'Neural network'):      # these need scaled inputs
        scaler = StandardScaler().fit(X_tr)                          # fit on the training window only
        X_tr, X_va, X_te = scaler.transform(X_tr), scaler.transform(X_va), scaler.transform(X_te)

    if model_name == 'Neural network':
        model = neural_network(X_tr.shape[1])
        model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=200, batch_size=64, verbose=0,
                  callbacks=[EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)])
        return [model.predict(X, verbose=0).ravel() for X in (X_tr, X_va, X_te)]

    if model_name == 'Logistic regression':
        model = LogisticRegression(max_iter=3000)
    elif model_name == 'Random forest':
        model = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42, n_jobs=-1)
    else:
        model = HistGradientBoostingClassifier(learning_rate=0.03, max_leaf_nodes=15, max_iter=100,
                                               min_samples_leaf=100, l2_regularization=1.0, random_state=42)
    model.fit(X_tr, y_tr)
    return [model.predict_proba(X)[:, 1] for X in (X_tr, X_va, X_te)]


# ---- Train every model on every window, score on train / validation / test ----
rows = []
for window, start in WINDOWS.items():
    train_set = full_df[full_df['race_year'].between(start, 2019)]
    X_tr, y_tr = train_set[MODEL_FEATURES], train_set['podium']
    X_va, y_va = val_set[MODEL_FEATURES], val_set['podium']
    X_te, y_te = test_set[MODEL_FEATURES], test_set['podium']

    for model_name in ['Logistic regression', 'Random forest', 'Gradient boosting', 'Neural network']:
        p_tr, p_va, p_te = train_and_predict(model_name, X_tr, y_tr, X_va, y_va, X_te)
        thr = best_threshold(y_va, p_va)                              # one threshold per model, from validation
        row = {'window': window, 'model': model_name, 'train rows': len(train_set)}
        for split, y, p in [('train', y_tr, p_tr), ('val', y_va, p_va), ('test', y_te, p_te)]:
            row[f'{split} PR-AUC']  = average_precision_score(y, p)
            row[f'{split} ROC-AUC'] = roc_auc_score(y, p)
            row[f'{split} F1']      = f1_score(y, (p >= thr).astype(int))
        rows.append(row)
    print('done:', window)

window_comparison = pd.DataFrame(rows).set_index(['model', 'window']).sort_index(level=0, sort_remaining=False)
window_comparison.round(3)

In [ ]:
# Compact view: validation and test PR-AUC per model (rows) and training window (columns)
for metric in ['val PR-AUC', 'test PR-AUC']:
    print(f'\n{metric}')
    display(window_comparison[metric].unstack('window')[list(WINDOWS)].round(3))

In [ ]:
import matplotlib.pyplot as plt

MODEL_ORDER  = ['Logistic regression', 'Random forest', 'Gradient boosting', 'Neural network']
MODEL_COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']          # fixed colour per model
METRICS = ['PR-AUC', 'ROC-AUC', 'F1']
SPLITS  = ['train', 'val', 'test']
windows = list(WINDOWS)
x = np.arange(len(windows))
width = 0.2                                                           # 4 bars side by side per window

fig, axes = plt.subplots(len(METRICS), len(SPLITS), figsize=(20, 13), sharex=True)
for r, metric in enumerate(METRICS):
    # same y-range for train / val / test of one metric, zoomed to the data so differences are visible
    values = window_comparison[[f'{s} {metric}' for s in SPLITS]]
    y_min, y_max = values.min().min() - 0.02, values.max().max() + 0.01
    for c, split in enumerate(SPLITS):
        ax = axes[r, c]
        for k, (model, color) in enumerate(zip(MODEL_ORDER, MODEL_COLORS)):
            scores = window_comparison.loc[model, f'{split} {metric}'].reindex(windows)
            ax.bar(x + (k - 1.5) * width, scores, width, color=color, edgecolor='white', linewidth=1.5,
                   label=model)
        ax.set_ylim(y_min, y_max)
        ax.set_title(f'{split} {metric}', fontsize=12)
        ax.set_xticks(x); ax.set_xticklabels(windows, fontsize=9)
        ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True)
        ax.spines[['top', 'right']].set_visible(False)
        if c == 0:
            ax.set_ylabel(f'{metric}  (axis does not start at 0)', fontsize=10)
    axes[-1, 1].set_xlabel('first training season (training always ends in 2019)')

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=4, fontsize=11, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.suptitle('Training-window comparison: same validation (2020-21) and test (2022-24) seasons for every window',
             fontsize=14, y=1.03)
plt.tight_layout(); plt.show()

In [ ]:
final_df = final_df[final_df['race_year'] >= 2003]

### Decision: train on 2003–2019
Evidence from the plots and tables above:
- **Data coverage:** 2003 is the first season with qualifying data for every race (1996–2002 only cover part of
  the races). Before 2003, qualifying positions are mostly copied from the grid.
- **Window comparison (validation 2020–2021):** logistic regression scores 0.742 with 2003+, against a best of
  0.744 (2012+). No window is clearly better.
- **Comparison against each model's best window (bootstrap):** 2003+ is **not significantly worse** than the best
  window for either model. 2004+ was significantly worse than 1994+ for gradient boosting, so 2004+ is ruled out.
- **All-pairs bootstrap:** only 1 of 56 comparisons was significant, which is in line with chance. Validation alone
  cannot rank the windows, so the choice uses the arguments above.
- **Trade-off:** 2003 adds one season of data (≈320 rows) compared with 2004. In 2003 the fastest-lap-rank
  feature has no data (filled with 20), but its 5-race average is real again from mid-2004.

**Choice: 2003+**, the earliest season with complete qualifying data and no significant loss against the best
window for any model.

### Final models: train 2003–2019, validate 2020–2021, test 2022–2024
Four models on the same 21 features. Logistic regression and the neural network use standardised inputs (scaler
fit on train only). The **neural network uses balanced class weights**: only ~15% of rows are podiums, so each
podium row counts ~3.6× during training. The F1 threshold of each model is chosen on validation and applied
unchanged to train and test.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'            # Keras on PyTorch (works locally and on Kaggle)
import keras
from keras import layers
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, precision_score,
                             recall_score, accuracy_score, log_loss, precision_recall_curve, confusion_matrix)

# ---- Data: train 2003-2019, validation 2020-2021, test 2022-2024 ----
START = 2003
train_set = full_df[full_df['race_year'].between(START, 2019)]
val_set   = full_df[full_df['race_year'].between(2020, 2021)]
test_set  = full_df[full_df['race_year'].between(2022, 2024)]
X_tr, y_tr = train_set[MODEL_FEATURES], train_set['podium']
X_va, y_va = val_set[MODEL_FEATURES],   val_set['podium']
X_te, y_te = test_set[MODEL_FEATURES],  test_set['podium']

# Scaled copies for logistic regression and the neural network (scaler fit on TRAIN only)
scaler = StandardScaler().fit(X_tr)
Xs_tr, Xs_va, Xs_te = scaler.transform(X_tr), scaler.transform(X_va), scaler.transform(X_te)


def best_threshold(y, prob):
    """Probability cut-off with the highest F1 -- chosen on VALIDATION only."""
    precision, recall, thresholds = precision_recall_curve(y, prob)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    return thresholds[np.argmax(f1[:-1])]


# ---- 1. Logistic regression ----
log_reg = LogisticRegression(max_iter=3000).fit(Xs_tr, y_tr)
probs = {'Logistic regression': [log_reg.predict_proba(X)[:, 1] for X in (Xs_tr, Xs_va, Xs_te)]}

# ---- 2. Random forest (grown in steps of 25 trees to record train/validation loss) ----
forest = RandomForestClassifier(n_estimators=25, min_samples_leaf=20, warm_start=True, random_state=42, n_jobs=-1)
rf_curve = {'trees': [], 'train': [], 'val': []}
for n_trees in range(25, 301, 25):
    forest.set_params(n_estimators=n_trees).fit(X_tr, y_tr)              # warm_start: adds trees, keeps old ones
    rf_curve['trees'].append(n_trees)
    rf_curve['train'].append(log_loss(y_tr, forest.predict_proba(X_tr)[:, 1]))
    rf_curve['val'].append(log_loss(y_va, forest.predict_proba(X_va)[:, 1]))
probs['Random forest'] = [forest.predict_proba(X)[:, 1] for X in (X_tr, X_va, X_te)]

# ---- 3. Gradient boosting (loss after every added tree) ----
boosting = HistGradientBoostingClassifier(learning_rate=0.03, max_leaf_nodes=15, max_iter=300,
                                          min_samples_leaf=100, l2_regularization=1.0, random_state=42)
boosting.fit(X_tr, y_tr)
gb_curve = {'train': [log_loss(y_tr, p[:, 1]) for p in boosting.staged_predict_proba(X_tr)],
            'val':   [log_loss(y_va, p[:, 1]) for p in boosting.staged_predict_proba(X_va)]}
probs['Gradient boosting'] = [boosting.predict_proba(X)[:, 1] for X in (X_tr, X_va, X_te)]

# ---- 4. Neural network: 2 hidden layers, balanced class weights ----
weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_tr)
class_weights = {0: weights[0], 1: weights[1]}                      # each podium row counts ~3.6x
print('class weights:', {k: round(v, 2) for k, v in class_weights.items()})

keras.utils.set_random_seed(42)
network = keras.Sequential([layers.Input(shape=(Xs_tr.shape[1],)),
                            layers.Dense(32, activation='relu'), layers.Dropout(0.2),
                            layers.Dense(16, activation='relu'), layers.Dropout(0.2),
                            layers.Dense(1, activation='sigmoid')])
network.compile(optimizer='adam', loss='binary_crossentropy',
                metrics=[keras.metrics.AUC(curve='PR', name='pr_auc')])
history = network.fit(Xs_tr, y_tr, validation_data=(Xs_va, y_va), epochs=200, batch_size=64,
                      class_weight=class_weights, verbose=0,
                      callbacks=[EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)])
probs['Neural network'] = [network.predict(X, verbose=0).ravel() for X in (Xs_tr, Xs_va, Xs_te)]

# ---- Scores: one threshold per model (from validation), applied to train / validation / test ----
thresholds, rows = {}, []
for model_name, (p_tr, p_va, p_te) in probs.items():
    thresholds[model_name] = best_threshold(y_va, p_va)
    row = {'model': model_name, 'threshold': thresholds[model_name]}
    for split, y, p in [('train', y_tr, p_tr), ('val', y_va, p_va), ('test', y_te, p_te)]:
        row[f'{split} PR-AUC']  = average_precision_score(y, p)
        row[f'{split} ROC-AUC'] = roc_auc_score(y, p)
        row[f'{split} F1']      = f1_score(y, (p >= thresholds[model_name]).astype(int))
    rows.append(row)
final_results = pd.DataFrame(rows).set_index('model')
final_results.round(3)

### Training vs validation curves
Loss is the error the model minimises (lower = better). The two lines should fall together; a validation line that
flattens or rises while the training line keeps falling means the model has started memorising the training seasons.
- **Neural network:** the training loss is *weighted* (class weights) and dropout is active during training, so its
  level is not directly comparable with validation; the trend is what matters. Early stopping keeps the best epoch.
- **Gradient boosting:** each added tree lowers the training loss; validation loss levels off, after which extra trees
  add nothing.
- **Random forest:** adding trees averages more votes but does not fit the data harder, so both lines stay flat.
- **Logistic regression** has no iterative curve (it is solved in one optimisation).

In [ ]:
# Training vs validation curves (lower loss = better; a widening gap = the model starts memorising)
fig, axes = plt.subplots(1, 4, figsize=(24, 4.5))
epochs = np.arange(1, len(history.history['loss']) + 1)
best_epoch = np.argmin(history.history['val_loss']) + 1

axes[0].plot(epochs, history.history['loss'], label='train'); axes[0].plot(epochs, history.history['val_loss'], label='validation')
axes[0].axvline(best_epoch, color='grey', linestyle='--', label=f'best epoch ({best_epoch}, weights kept)')
axes[0].set(title='Neural network: loss per epoch', xlabel='epoch', ylabel='weighted binary cross-entropy')

axes[1].plot(epochs, history.history['pr_auc'], label='train'); axes[1].plot(epochs, history.history['val_pr_auc'], label='validation')
axes[1].axvline(best_epoch, color='grey', linestyle='--')
axes[1].set(title='Neural network: PR-AUC per epoch', xlabel='epoch', ylabel='PR-AUC')

axes[2].plot(range(1, len(gb_curve['train']) + 1), gb_curve['train'], label='train')
axes[2].plot(range(1, len(gb_curve['val']) + 1), gb_curve['val'], label='validation')
axes[2].set(title='Gradient boosting: loss per added tree', xlabel='number of trees', ylabel='log-loss')

axes[3].plot(rf_curve['trees'], rf_curve['train'], marker='o', label='train')
axes[3].plot(rf_curve['trees'], rf_curve['val'], marker='o', label='validation')
axes[3].set(title='Random forest: loss vs number of trees', xlabel='number of trees', ylabel='log-loss')

for ax in axes:
    ax.legend(); ax.grid(alpha=0.3); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

### Neural network (balanced class weights): confusion matrix and metrics
Yes/no decisions use the threshold chosen on validation. With class weights the predicted probabilities are inflated
(podiums were up-weighted), so the chosen threshold is higher than for the other models. Accuracy is shown for
completeness but is not used to judge the model: predicting "no podium" for everyone already scores ~85%.

In [ ]:
# Neural network (balanced class weights): confusion matrices and classification metrics
thr_nn = thresholds['Neural network']
p_tr, p_va, p_te = probs['Neural network']
labels = ['no podium', 'podium']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
nn_metrics = []
for ax, (split, y, p) in zip(axes, [('Validation 2020-21', y_va, p_va), ('Test 2022-24', y_te, p_te)]):
    pred = (p >= thr_nn).astype(int)
    sns.heatmap(confusion_matrix(y, pred), annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=labels, yticklabels=labels, ax=ax, annot_kws={'size': 14})
    ax.set(title=f'Neural network - {split}', xlabel='Predicted', ylabel='Actual')
plt.tight_layout(); plt.show()

for split, y, p in [('train', y_tr, p_tr), ('validation', y_va, p_va), ('test', y_te, p_te)]:
    pred = (p >= thr_nn).astype(int)
    nn_metrics.append({'split': split, 'precision': precision_score(y, pred), 'recall': recall_score(y, pred),
                       'F1': f1_score(y, pred), 'accuracy': accuracy_score(y, pred)})
print(f'threshold = {thr_nn:.3f} (chosen on validation)')
pd.DataFrame(nn_metrics).set_index('split').round(3)

## Explainability (XAI)
We explain the gradient-boosting model on the **test seasons (2022–2024)**.
- **Global:** permutation importance (how much test PR-AUC drops when one feature is shuffled) and SHAP beeswarm
  plots (how each feature pushes every prediction up or down).
- **Local:** one prediction the model got **right** and one it got **wrong**, each explained with SHAP and LIME.

**Important:** these explanations show what the model *relies on*, not what *causes* a podium. A feature can be
important because it is correlated with the real cause (e.g. car speed), not because it causes the result.

In [ ]:
from sklearn.inspection import permutation_importance

# Permutation importance: shuffle one feature at a time on the TEST set and measure how much PR-AUC drops
perm = permutation_importance(boosting, X_te, y_te, scoring='average_precision', n_repeats=10, random_state=42)
perm_importance = pd.Series(perm.importances_mean, index=MODEL_FEATURES).sort_values()

perm_importance.plot(kind='barh', figsize=(8, 7), color='#2a78d6')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Permutation importance (gradient boosting, test 2022-2024)')
plt.xlabel('drop in PR-AUC when the feature is shuffled')
plt.tight_layout(); plt.show()

### SHAP beeswarm (global)
Each dot is one test row. Position on the x-axis = how much that feature pushed the prediction towards podium
(right) or away from it (left), in log-odds. Colour = the feature's value (red high, blue low).

In [ ]:
import shap

# Model to explain: gradient boosting (tree model -> exact and fast SHAP values with TreeExplainer)
model_to_explain = boosting
explainer = shap.TreeExplainer(model_to_explain)
shap_test = explainer(X_te)                           # SHAP values for every test row (in log-odds units)

# 1. Global: beeswarm for the class "podium"
shap.plots.beeswarm(shap_test, max_display=21, show=False)
plt.title('SHAP beeswarm - class "podium" (gradient boosting, test 2022-2024)')
plt.tight_layout(); plt.show()

# 2. Global: beeswarm for the class "no podium" (binary task: exactly the mirror image of the podium plot)
shap_no_podium = shap.Explanation(values=-shap_test.values, base_values=-shap_test.base_values,
                                 data=shap_test.data, feature_names=shap_test.feature_names)
shap.plots.beeswarm(shap_no_podium, max_display=21, show=False)
plt.title('SHAP beeswarm - class "no podium" (mirror of the podium plot)')
plt.tight_layout(); plt.show()

### Choosing the records for local explanations
- **Right:** the test row with the highest podium probability that really finished on the podium (true positive).
- **Wrong:** the test row with the lowest podium probability that still finished on the podium (missed podium).

In [ ]:
# Pick two test records: one the model got RIGHT and one it got WRONG
p_test = model_to_explain.predict_proba(X_te)[:, 1]
pred_test = (p_test >= thresholds['Gradient boosting']).astype(int)
records = test_set[['race_year', 'race_round', 'driverId', 'constructor_name', 'grid_pos', 'quali_position', 'podium']].copy()
records['podium probability'] = p_test.round(3)
records['predicted podium'] = pred_test

right = np.where((pred_test == 1) & (y_te.values == 1))[0]          # podium predicted AND podium happened
wrong = np.where((pred_test == 0) & (y_te.values == 1))[0]          # podium happened but the model said no (missed)
idx_right = right[np.argmax(p_test[right])]                          # the most confident correct podium
idx_wrong = wrong[np.argmin(p_test[wrong])]                          # the most surprising missed podium
records.iloc[[idx_right, idx_wrong]].assign(case=['RIGHT (true positive)', 'WRONG (missed podium)'])

In [ ]:
clean=pd.read_csv("C:\\Users\\amiro\\ACL_9th-F1\\F1_clean\\all_features.csv")
clean = clean[ clean['race_year'] (clean['driverId'] == 842)]

clean.info()

### SHAP local explanations
- **Force plot (right prediction):** starts at the base value (the average prediction) and shows which features push
  this driver's prediction up (red) or down (blue) to the final value f(x).
- **Waterfall (wrong prediction):** the same idea step by step, largest contributions first.

In [ ]:
# Same SHAP values, feature values rounded to 2 decimals so the labels stay readable
shap_display = shap.Explanation(values=shap_test.values, base_values=shap_test.base_values,
                                data=np.round(shap_test.data.astype(float), 2), feature_names=shap_test.feature_names)

# 3. Local: force plot for the CORRECT prediction
shap.plots.force(shap_display[idx_right], matplotlib=True, show=False, figsize=(22, 3.5), text_rotation=15)
plt.gcf().suptitle('SHAP force plot - correct prediction (true positive)', y=1.25, fontsize=13)
plt.show()
# 4. Local: waterfall plot for the WRONG prediction
shap.plots.waterfall(shap_display[idx_right], max_display=12, show=False)
plt.title('SHAP waterfall -right prediction (missed podium)')
plt.tight_layout(); plt.show()
# 4. Local: waterfall plot for the WRONG prediction
shap.plots.waterfall(shap_display[idx_wrong], max_display=12, show=False)
plt.title('SHAP waterfall - wrong prediction (missed podium)')
plt.tight_layout(); plt.show()

In [ ]:
# Two more NEGATIVE predictions (model said "no podium")
false_pos = np.where((pred_test == 1) & (y_te.values == 0))[0]     # said no, and no podium happened
    # said no, but podium happened
idx_tn   = false_pos[np.argmin(p_test[false_pos])]                  # most confident correct "no podium"                      # missed podium closest to the threshold

print('threshold:', round(thresholds['Gradient boosting'], 3))
records.iloc[[idx_tn, idx_near]].assign(case=['RIGHT (true negative)', 'WRONG (near-miss podium)'])

In [ ]:
for idx, case in [(idx_tn, 'correct "no podium" (true negative)'), (idx_near, 'wrong - near-miss podium (false negative)')]:
    shap.plots.waterfall(shap_display[idx], max_display=12, show=False)
    plt.title(f'SHAP waterfall - {case}\nmodel podium probability = {p_test[idx]:.3f}')
    plt.tight_layout(); plt.show()

In [ ]:
# Two more NEGATIVE predictions (model said "no podium")
true_neg = np.where((pred_test == 0) & (y_te.values == 0))[0]     # said no, and no podium happened
missed   = np.where((pred_test == 0) & (y_te.values == 1))[0]     # said no, but podium happened
idx_tn   = true_neg[np.argmin(p_test[true_neg])]                  # most confident correct "no podium"
idx_near = missed[np.argmax(p_test[missed])]                      # missed podium closest to the threshold

print('threshold:', round(thresholds['Gradient boosting'], 3))
records.iloc[[idx_tn, idx_near]].assign(case=['RIGHT (true negative)', 'WRONG (near-miss podium)'])

In [ ]:
# SHAP waterfall for both negative examples
for idx, case in [(idx_tn, 'correct "no podium" (true negative)'), (idx_near, 'wrong - near-miss podium (false negative)')]:
    shap.plots.waterfall(shap_display[idx], max_display=12, show=False)
    plt.title(f'SHAP waterfall - {case}\nmodel podium probability = {p_test[idx]:.3f}')
    plt.tight_layout(); plt.show()

In [ ]:
import lime
import lime.lime_tabular

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_tr),              # Use your training feature matrix (as a numpy array)
    feature_names=MODEL_FEATURES,             # List of your feature names
    class_names=['Not Selected', 'Selected'], # Change to match your target classes (0 and 1)
    mode='classification'                     # Use 'regression' if predicting a continuous number
)

for idx, case in [(idx_tn, 'correct "no podium" (true negative)'), (idx_near, 'wrong - near-miss podium (false negative)')]:
    explanation = lime_explainer.explain_instance(X_te.values[idx], model_to_explain.predict_proba, num_features=10)
    explanation.as_pyplot_figure(label=1)
    plt.title(f'LIME - {case}\nmodel podium probability = {p_test[idx]:.2f}')
    plt.tight_layout(); plt.show()

## Feature-group ablation
We remove **one group of features at a time**, retrain all four models with the same settings, seasons and
validation-chosen thresholds, and compare against the model that uses all 21 features.
- **Δ PR-AUC < 0:** the model is worse without the group, so the group was helping.
- **Δ PR-AUC ≈ 0:** the other groups already carry the same information.
- **Decision rule (stated before looking):** decisions are made on **validation** (2020–2021). Test is only reported.
  A group counts as useful if removing it lowers validation PR-AUC by more than 0.01 for most models.

In [ ]:
# ---- Feature-group ablation: drop one group, retrain all 4 models, compare with the full model ----
FEATURE_GROUPS = {
    'Grid & qualifying':        ['grid_pos', 'quali_position', 'grid_penalty_delta', 'outqualified_teammate'],
    'Driver recent form':       ['driver_avg_pos_last_5', 'driver_avg_lap_pos_last_5', 'driver_avg_speed_rank_last_5',
                                 'driver_prev_position', 'driver_prev_podium', 'won_previous_race'],
    'Driver career & circuit':  ['driver_career_avg_pos', 'driver_career_podiums', 'driver_career_races',
                                 'driver_avg_pos_at_circuit'],
    'Championship standing':    ['points_share', 'pre_race_wins', 'constructor_points_share', 'pre_race_constructor_wins'],
    'Team (constructor) form':  ['constructor_prev_pos', 'constructor_avg_pos_last_5', 'constructor_career_podiums'],
    'result_from_Xai':          ['constructor_avg_pos_last_5', 'quali_position']}

# every feature belongs to exactly one group
# every model feature is in at least one group (the XAI group overlaps the others on purpose)
assert set(MODEL_FEATURES) <= set(sum(FEATURE_GROUPS.values(), []))

def make_network(n_inputs):
    """Same shallow network as in training: 2 hidden layers."""
    keras.utils.set_random_seed(42)
    net = keras.Sequential([layers.Input(shape=(n_inputs,)),
                            layers.Dense(32, activation='relu'), layers.Dropout(0.2),
                            layers.Dense(16, activation='relu'), layers.Dropout(0.2),
                            layers.Dense(1, activation='sigmoid')])
    net.compile(optimizer='adam', loss='binary_crossentropy')
    return net


def train_and_score(features):
    """Train the 4 models on `features` (same settings as the final training) and score validation + test."""
    Xtr, Xva, Xte = train_set[features], val_set[features], test_set[features]
    sc = StandardScaler().fit(Xtr)                                   # scaler fit on TRAIN only
    Str, Sva, Ste = sc.transform(Xtr), sc.transform(Xva), sc.transform(Xte)

    lr = LogisticRegression(max_iter=3000).fit(Str, y_tr)
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42, n_jobs=-1).fit(Xtr, y_tr)
    gb = HistGradientBoostingClassifier(learning_rate=0.03, max_leaf_nodes=15, max_iter=300,
                                        min_samples_leaf=100, l2_regularization=1.0, random_state=42).fit(Xtr, y_tr)
    nn = make_network(len(features))
    nn.fit(Str, y_tr, validation_data=(Sva, y_va), epochs=200, batch_size=64, class_weight=class_weights, verbose=0,
           callbacks=[EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)])

    model_probs = {'Logistic regression': (lr.predict_proba(Sva)[:, 1], lr.predict_proba(Ste)[:, 1]),
                   'Random forest':       (rf.predict_proba(Xva)[:, 1], rf.predict_proba(Xte)[:, 1]),
                   'Gradient boosting':   (gb.predict_proba(Xva)[:, 1], gb.predict_proba(Xte)[:, 1]),
                   'Neural network':      (nn.predict(Sva, verbose=0).ravel(), nn.predict(Ste, verbose=0).ravel())}
    scores = []
    for model_name, (p_va, p_te) in model_probs.items():
        thr = best_threshold(y_va, p_va)                             # threshold from VALIDATION only
        scores.append({'model': model_name,
                       'val PR-AUC':   average_precision_score(y_va, p_va),
                       'test PR-AUC':  average_precision_score(y_te, p_te),
                       'test ROC-AUC': roc_auc_score(y_te, p_te),
                       'test F1':      f1_score(y_te, (p_te >= thr).astype(int))})
    return scores


# 1 run with all 21 features + 1 run per dropped group
runs = {'None (all 21 features)': MODEL_FEATURES}
for group, cols in FEATURE_GROUPS.items():
    runs[group] = [f for f in MODEL_FEATURES if f not in cols]

rows = []
for dropped, features in runs.items():
    for score in train_and_score(features):
        rows.append({'dropped group': dropped, 'features left': len(features), **score})
    print('done:', dropped)

ablation = pd.DataFrame(rows)
# change vs the full model (same model): negative = the model got WORSE without that group
full = ablation[ablation['dropped group'] == 'None (all 21 features)'].set_index('model')
for metric in ['val PR-AUC', 'test PR-AUC']:
    ablation[f'Δ {metric}'] = ablation[metric] - ablation['model'].map(full[metric])
ablation.round(3)

- **result_from_Xai:** the two features SHAP ranked highest (`constructor_avg_pos_last_5`, `quali_position`), removed
  together to check whether the model really depends on them. This group overlaps the groups above.

In [ ]:
# Bar chart: how much PR-AUC each model LOSES when a group is removed (below 0 = the group was helping)
dropped_only = ablation[ablation['dropped group'] != 'None (all 21 features)']
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, metric in zip(axes, ['Δ val PR-AUC', 'Δ test PR-AUC']):
    table = dropped_only.pivot(index='dropped group', columns='model', values=metric)[list(probs)]
    table.plot(kind='bar', ax=ax, color=['#2a78d6', '#eb6834', '#1baf7a', '#eda100'], width=0.8, legend=False)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'{metric} after removing the group'); ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)
axes[0].set_ylabel('change vs all 21 features')
axes[1].legend(title='model', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout(); plt.show()